# English Edition — from-scratch GPT (~124M) on 2× Tesla T4

This notebook runs the **English edition** of the `myAI` repository. It trains
an English GPT-style model **from scratch** on streaming
[FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) and
then fine-tunes it for chat.

* Model: **124,278,528** parameters (14 layers, 768 wide, 12 heads, RoPE,
  SwiGLU, RMSNorm, tied embeddings, context 1,024).
* Tokenizer: 32K SentencePiece BPE trained on a bounded English sample.
* No legacy Vietnamese source, tokenizer, checkpoint or dataset is included.

**Run All is safe.** Real pretraining, tokenizer training and SFT are gated by
flags that default to `False`. The smoke tests always run and write only to
`tokenizer_en/smoke/` and `checkpoints_en/smoke/`.

Before running: **Settings → Accelerator → GPU T4 ×2** and
**Settings → Internet → On**. See `KAGGLE_EN.md` for the full guide.

## 1. Locate the repository

If the repository is already in `/kaggle/working` (for example attached as a
Kaggle Dataset or uploaded by hand), this finds it. Otherwise set `REPO_URL`
to a git URL and the notebook clones it.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Optional: set to a git URL if the repository is not already present.
REPO_URL = ""

candidates = [
    Path("/kaggle/working/myai-en"),
    Path("/kaggle/working/test-AI"),
    Path.cwd(),
    Path("/kaggle/working"),
]
ROOT = next((p for p in candidates if (p / "english" / "train.py").exists()), None)
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/myai-en"], check=True)
    ROOT = Path("/kaggle/working/myai-en")
if ROOT is None:
    raise SystemExit(
        "Repository not found. Upload it to /kaggle/working or set REPO_URL above."
    )

os.chdir(ROOT)
os.environ["HF_HOME"] = "/kaggle/working/.hf"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

print("repo     :", ROOT)
print("python   :", sys.version.split()[0])
print("HF_HOME  :", os.environ["HF_HOME"])
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found (CPU session?)")

## 2. Install dependencies

The Kaggle image already ships a CUDA build of PyTorch, so `torch` is left
unpinned and normally stays as-is. `datasets` + `pyarrow` are required for
streaming FineWeb-Edu parquet shards.

In [ ]:
%pip install -q -r requirements.txt
print("dependencies installed")

## 3. Verify the GPUs

You should see **two Tesla T4** devices. `english.train` refuses distributed
training without CUDA/NCCL, which is intentional: DDP on CPU is not supported.

In [ ]:
import torch

print("torch        :", torch.__version__)
print("cuda build   :", torch.version.cuda)
print("available    :", torch.cuda.is_available())
print("device count :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"gpu{i}         : {p.name}, {p.total_memory / 2**30:.1f} GiB, cc{p.major}.{p.minor}")
if torch.cuda.device_count() != 2:
    print("\nWARNING: select Settings -> Accelerator -> GPU T4 x2 before training.")

## 4. Smoke test (always runs)

This trains a 512-token smoke tokenizer and a **393,856**-parameter model for
a few steps under `tokenizer_en/smoke/` and `checkpoints_en/smoke/`. It does not
touch real artifacts and takes about a minute.

In [ ]:
import subprocess
import sys

def run(*args):
    printable = " ".join(str(a) for a in args)
    print("$", printable, flush=True)
    subprocess.run([str(a) for a in args], check=True)

run(sys.executable, "-m", "english.train", "--smoke-test", "--max-steps", "4")
run(sys.executable, "-m", "english.sft", "--smoke-test", "--max-steps", "2")
run(
    sys.executable, "-m", "english.chat", "--smoke-test",
    "--prompt", "Hello", "--max-new-tokens", "24", "--no-rag", "--no-web",
)
print("\nsmoke test complete")

## 5. CPU test suite (optional)

118 tests with no network access. On Kaggle this takes a couple of minutes.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "english/tests", "-t", ".", "-v"],
    check=True,
)

## 6. Train the real 32K tokenizer (once)

Streams a bounded 200 MB English sample, trains a SentencePiece BPE model and
validates the fixed special ids (`pad=0, unk=1, bos=2, eos=3, user=4,
assistant=5, system=6`). The sample is deleted afterwards.

Set `RUN_TOKENIZER_TRAIN = True` and run this cell. **Do not retrain the
tokenizer** for a run you intend to resume: checkpoints store its SHA-256 and
refuse a mismatch.

In [ ]:
import subprocess
import sys
from pathlib import Path

RUN_TOKENIZER_TRAIN = False
TOKENIZER = Path("tokenizer_en/tokenizer.model")

if TOKENIZER.exists():
    print("tokenizer already exists:", TOKENIZER.resolve())
    subprocess.run([sys.executable, "-m", "english.tokenizer", "--inspect"], check=True)
elif RUN_TOKENIZER_TRAIN:
    subprocess.run(
        [sys.executable, "-m", "english.tokenizer", "--train",
         "--sample-mb", "200", "--vocab-size", "32768"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "english.tokenizer", "--inspect"], check=True)
else:
    print("Set RUN_TOKENIZER_TRAIN = True to train tokenizer_en/tokenizer.model")

## 7. Pretrain on FineWeb-Edu

`RUN_REAL_PRETRAIN` is `False` so **Run All never starts training**. When you
are ready, set it to `True` and run this cell.

* `PLAN_TOKENS = 300_000_000` is a good first run (roughly 7–10 h on 2×T4).
* `tokens/step = 2 GPUs × 8 × 4 × 1024 = 65,536`.
* Automatic resume: if `checkpoints_en/pretrain_last.pth` exists, the same
  command continues from its step, tokens, optimizer, scaler, RNG and data
  position.
* Free Kaggle quota is about **30 GPU-hours/week** and both T4s count. Start
  small; see `KAGGLE_EN.md` for the time table.

In [ ]:
import os
import shutil
import subprocess

RUN_REAL_PRETRAIN = False
PLAN_TOKENS = 300_000_000
TOKENIZER = "tokenizer_en/tokenizer.model"
EXPECTED_PARAMS = 124_278_528

torchrun = shutil.which("torchrun") or "torchrun"
cmd = [
    torchrun, "--nproc_per_node=2", "-m", "english.train",
    "--preset", "base",
    "--plan-tokens", str(PLAN_TOKENS),
    "--batch-size", "8",
    "--grad-accum", "4",
    "--amp", "fp16",
    "--expect-params", str(EXPECTED_PARAMS),
    "--eval-every", "250",
    "--save-every", "250",
    "--log-every", "10",
]
print("$", " ".join(cmd), flush=True)

if not os.path.exists(TOKENIZER):
    print("\nTokenizer missing. Run the tokenizer cell first.")
elif not RUN_REAL_PRETRAIN:
    print("\nRUN_REAL_PRETRAIN is False. Set it to True and re-run this cell to train.")
else:
    subprocess.run(cmd, check=True)

## 8. Resume in a later session

Run the pretraining cell again (same command) and resume is automatic. Use the
explicit form below if you prefer. Expect `data=exact` in the log; if you
changed the GPU count, exact resume is impossible and
`--allow-data-restart` re-reads the stream from the start.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

RUN_RESUME = False
last = Path("checkpoints_en/pretrain_last.pth")
torchrun = shutil.which("torchrun") or "torchrun"
cmd = [
    torchrun, "--nproc_per_node=2", "-m", "english.train",
    "--preset", "base",
    "--plan-tokens", "300000000",
    "--batch-size", "8", "--grad-accum", "4",
    "--amp", "fp16",
    "--expect-params", "124278528",
    "--eval-every", "250", "--save-every", "250", "--log-every", "10",
    "--resume", "auto",
]
print("$", " ".join(cmd), flush=True)

if not last.exists():
    print("\nNo checkpoints_en/pretrain_last.pth yet. Run pretraining first.")
elif not RUN_RESUME:
    print("\nRUN_RESUME is False. Set it to True and re-run this cell to continue.")
else:
    subprocess.run(cmd, check=True)

## 9. Supervised fine-tuning (SFT)

Assistant-only loss: system prompts, role headers, user turns and padding are
masked with `-100`. Uses the bundled `data/sft_sample_en.jsonl` (43 examples)
by default; replace it with a larger English instruction file for a useful
assistant.

Set `RUN_SFT = True` and run this cell. Resume with the same cell
(`sft_last.pth` is auto-detected).

In [ ]:
import subprocess
import sys
from pathlib import Path

RUN_SFT = False
candidates = [
    Path("checkpoints_en/pretrain_best.pth"),
    Path("checkpoints_en/pretrain_last.pth"),
]
init = next((p for p in candidates if p.exists()), None)
cmd = [
    sys.executable, "-m", "english.sft",
    "--init", str(init) if init else "",
    "--data", "data/sft_sample_en.jsonl",
    "--epochs", "3",
    "--batch-size", "4",
    "--amp", "auto",
    "--val-ratio", "0.1",
]
print("$", " ".join(cmd), flush=True)

if init is None:
    print("\nNo pretrained checkpoint found. Run pretraining first.")
elif not RUN_SFT:
    print("\nRUN_SFT is False. Set it to True and re-run this cell to fine-tune.")
else:
    subprocess.run(cmd, check=True)

## 10. Chat

Runs one-shot and non-interactive so it also works in a committed notebook.
Prefers the SFT checkpoint, then falls back to the pretrain checkpoint, then to
the smoke checkpoint. Web search is disabled here; enable it by passing
`--show-sources` and removing `--no-web` (Internet must be on).

In [ ]:
import subprocess
import sys
from pathlib import Path

options = [
    Path("checkpoints_en/sft_best.pth"),
    Path("checkpoints_en/sft_last.pth"),
    Path("checkpoints_en/pretrain_best.pth"),
    Path("checkpoints_en/pretrain_last.pth"),
    Path("checkpoints_en/smoke/sft_best.pth"),
    Path("checkpoints_en/smoke/pretrain_best.pth"),
]
checkpoint = next((p for p in options if p.exists()), None)

if checkpoint is None:
    print("No checkpoint available. Run the smoke test or training first.")
else:
    tokenizer = (
        "tokenizer_en/smoke/tokenizer.model"
        if "smoke" in checkpoint.parts
        else "tokenizer_en/tokenizer.model"
    )
    cmd = [
        sys.executable, "-m", "english.chat",
        "--checkpoint", str(checkpoint),
        "--tokenizer", tokenizer,
        "--prompt", "Explain what a transformer is in two sentences.",
        "--max-new-tokens", "128",
        "--no-web",
    ]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

## 11. Save artifacts between sessions

`/kaggle/working` is only fully reliable when you commit the notebook. This
cell copies checkpoints, tokenizer, logs and the validation bin into
`/kaggle/working/artifacts_en/` and creates a ZIP.

Then use **Save Version → Save & Run All (Commit)**, create a **Kaggle
Dataset** from the output, and attach it in the next session.

In [ ]:
import shutil
from pathlib import Path

ART = Path("/kaggle/working/artifacts_en")
if ART.exists():
    shutil.rmtree(ART)
ART.mkdir(parents=True)

for name in ("checkpoints_en", "tokenizer_en", "runs_en", "data_en"):
    src = Path(name)
    if src.exists():
        print("copying", src)
        shutil.copytree(src, ART / name, dirs_exist_ok=True)

archive = shutil.make_archive("/kaggle/working/myai-en-artifacts", "zip", ART)
size = Path(archive).stat().st_size / 2**30
print(f"\nwrote {archive} ({size:.2f} GiB)")

## 12. Restore artifacts in a new session

Attach the artifacts Dataset, then set `SRC` to its path under
`/kaggle/input/...` and run this cell. It copies the artifacts back into the
repository before you resume training.

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/myai-en-artifacts/artifacts_en")  # adjust to the attached dataset

if not SRC.exists():
    print("Artifacts not found at", SRC)
    print("Attach the artifacts Kaggle Dataset and update SRC above.")
else:
    for name in ("checkpoints_en", "tokenizer_en", "runs_en", "data_en"):
        src = SRC / name
        if src.exists():
            print("restoring", src)
            shutil.copytree(src, Path(name), dirs_exist_ok=True)
    print("\nrestored. Resume with the pretraining cell.")

## 13. Notes and limitations

* The repository ships **no trained weights**. Everything here starts from
  random initialization; `smoke` checkpoints are only 393K-parameter test
  artifacts.
* Exact data resume requires the same dataset source, tokenizer, `block_size`
  and **world size**. Changing the GPU count needs `--allow-data-restart` and
  is not exact.
* T4 (Turing) has no native BF16. Use `--amp fp16` or `--amp auto`; forcing
  `--amp bf16` raises an error.
* The bundled SFT sample is tiny. Extend it before expecting a capable
  assistant.
* A 5B-token run is a multi-week project on free Kaggle quota. Start with
  100M–300M tokens and scale only if the loss curve and quota allow it.
* If HF shard cache fills the disk, delete `/kaggle/working/.hf`; the next run
  re-downloads shards.